# Cross-machine results consolidation & agreement check

**Read-only over `results/`.** Unions every machine-stamped result CSV (and handshake pilot),
tags each row with its source `MACHINE_ID`, prints a per-notebook/scheme completion matrix
(horizons × machines), and cross-checks fits present on more than one machine.

**Contract.** `environment.yml` is now **OpenBLAS-pinned**, so across OS/BLAS the contract is
**numerical tolerance**, not bit-equality: numeric metrics + `tau_w` must agree within
`np.isclose(rtol=1e-6, atol=1e-9)`. (The pure-digest fingerprints — `config.hash16` /
`feature_spec.hash16` — must still be bit-equal; that is the separate handshake gate.
`config_hash` is also checked here for exact equality as a bonus.)

This notebook never fits or writes — it only reads and reports.

In [ ]:
import os, re, glob
import numpy as np
import pandas as pd

# Repo-root resolver (works whether cwd is the repo root or notebooks/).
_ROOT = os.getcwd()
while _ROOT != os.path.dirname(_ROOT) and not os.path.isdir(os.path.join(_ROOT, 'src')):
    _ROOT = os.path.dirname(_ROOT)

# Cross-machine numeric agreement tolerance (OpenBLAS-pinned env -> tolerance, not bit-equality).
RTOL, ATOL = 1e-6, 1e-9

# Search roots: canonical repo-root results/, plus notebooks/results/ (where notebooks write
# when launched from notebooks/). Both are scanned; missing roots are skipped.
RESULT_ROOTS = [os.path.join(_ROOT, 'results'), os.path.join(_ROOT, 'notebooks', 'results')]

# Machine-stamped sweep outputs + handshake pilots.
PATTERNS = [
    'pooled_v5/pooled_metrics_wf_H*__*.csv',
    'pooled_v5/pooled_metrics_blockcv_H*__*.csv',
    'v5_nor/v5_wf_H*__*.csv',
    'v5_metrics_long__*.csv',
    'pooled_v5/_pilot/*/pilot_pooled_wf_H*.csv',   # pooled handshake pilots
    'v5_nor/_pilot/*/pilot_metrics_H*.csv',        # per-security handshake pilots
]
print('repo root   :', _ROOT)
print('result roots:', [os.path.relpath(r, _ROOT) for r in RESULT_ROOTS if os.path.isdir(r)])

In [ ]:
def _machine_from_path(path):
    """MACHINE_ID from a sweep filename (...__<machine>.csv) or a pilot dir (.../<tag>__<machine>/)."""
    stem = os.path.basename(path)[:-4] if path.endswith('.csv') else os.path.basename(path)
    m = re.search(r'__([A-Za-z0-9_.-]+)$', stem)        # sweep file: name__machine
    if m:
        return m.group(1)
    parent = os.path.basename(os.path.dirname(path))     # pilot dir: tag__machine
    return parent.split('__', 1)[1] if '__' in parent else 'unknown'

def _notebook_from_path(path):
    p = path.replace(os.sep, '/')
    return 'pooled_v5' if 'pooled_v5' in p else ('v5_nor' if 'v5_nor' in p else 'unknown')

frames, report = [], []
for root in RESULT_ROOTS:
    for pat in PATTERNS:
        for path in sorted(glob.glob(os.path.join(root, pat))):
            try:
                d = pd.read_csv(path)                     # tolerate partial/locked/unreadable
            except Exception as e:
                report.append((path, 0, f'SKIP ({type(e).__name__}: {str(e)[:38]})')); continue
            if d.empty:
                report.append((path, 0, 'empty')); continue
            d['source_machine']  = _machine_from_path(path)
            d['source_notebook'] = _notebook_from_path(path)
            d['source_file']     = os.path.relpath(path, _ROOT)
            frames.append(d); report.append((path, len(d), 'OK'))

df_all = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f'loaded {sum(1 for _,_,s in report if s=="OK")} file(s), {len(df_all)} row(s); '
      f'machines: {sorted(df_all["source_machine"].unique()) if len(df_all) else []}')
for path, n, st in report:
    print(f'  {st:<28} {n:>5}  {os.path.relpath(path, _ROOT)}')

In [ ]:
# Completion matrix: horizons x machines (count of distinct folds present) per notebook/scheme.
if df_all.empty:
    print('no result rows loaded — nothing to summarise yet.')
else:
    has_cats = 'use_categoricals' in df_all.columns
    for (nbk, scheme), sub in df_all.groupby(['source_notebook', 'validation_scheme'], dropna=False):
        idx = ['horizon'] + (['use_categoricals']
                             if has_cats and sub['use_categoricals'].notna().any() else [])
        mat = sub.pivot_table(index=idx, columns='source_machine',
                              values='fold', aggfunc='nunique', fill_value=0)
        print(f'\n=== completion: {nbk} / {scheme}  (distinct folds, rows=horizon[xcats] x machine) ===')
        print(mat)

In [ ]:
# Cross-machine agreement on fits present on >1 machine.
KEY_CANDIDATES = ['source_notebook', 'validation_scheme', 'security', 'country', 'tenor',
                  'fold', 'regime', 'sample', 'horizon', 'agg_level', 'use_categoricals',
                  'model_kind', 'is_pooled']
METRIC_CANDIDATES = ['n_obs', 'dir_acc', 'r2_raw', 'mse_model', 'mse_rw', 'ct_r2_oos',
                     'cw_stat', 'cw_pval', 'dmh_stat', 'dmh_pval', 'pt_stat', 'pt_pval',
                     'binom_pval', 'n_eff', 'pca_k', 'xm_pca_evr', 'dir_coverage', 'tau_w',
                     'feature_count', 'config_hash']

if df_all.empty:
    print('AGREEMENT: N/A (no rows loaded)')
else:
    keys    = [c for c in KEY_CANDIDATES if c in df_all.columns]
    metrics = [c for c in METRIC_CANDIDATES if c in df_all.columns]
    obj_m   = [m for m in metrics if df_all[m].dtype == object]      # e.g. config_hash -> exact
    num_m   = [m for m in metrics if m not in obj_m]
    flags, n_num_cmp, n_shared = [], 0, 0
    for kv, grp in df_all.groupby(keys, dropna=False):
        if grp['source_machine'].nunique() < 2:
            continue
        n_shared += 1
        kd = dict(zip(keys, kv if isinstance(kv, tuple) else (kv,)))
        for m in obj_m:
            u = grp[m].dropna().astype(str).unique()
            if len(u) > 1:
                flags.append((kd, m, 'EXACT-MISMATCH',
                              grp.groupby('source_machine')[m].first().to_dict()))
        for m in num_m:
            v = grp[['source_machine', m]].dropna()
            if v['source_machine'].nunique() < 2:
                continue
            n_num_cmp += 1
            arr = v[m].to_numpy(float)
            if not np.all(np.isclose(arr, arr[0], rtol=RTOL, atol=ATOL, equal_nan=True)):
                mad = float(np.nanmax(np.abs(arr - arr[0])))
                flags.append((kd, m, f'max|d|={mad:.3e}',
                              v.set_index('source_machine')[m].to_dict()))
    print(f'fit-keys on >1 machine: {n_shared};  numeric comparisons: {n_num_cmp};  flagged: {len(flags)}')
    if n_shared == 0:
        print('  (no fit-key present on >1 machine yet — nothing to cross-check; load more machines\' files)')
    for kd, m, info, vals in flags[:50]:
        kshort = {k: kd[k] for k in ('source_notebook','validation_scheme','horizon','fold','security')
                  if k in kd}
        print(f'  FLAG [{m}] {info}  {kshort}  {vals}')
    print(f'\nAGREEMENT: {"PASS" if not flags else "FAIL"}   '
          f'(RTOL={RTOL}, ATOL={ATOL}; {n_shared} shared fit-key(s), {len(flags)} flag(s))')

## Interpreting the result

- **Completion matrix** — which (horizon × machine) arms have landed, per notebook/scheme.
  Zeros = not yet run on that machine.
- **`AGREEMENT: PASS`** — every fit present on >1 machine agrees within tolerance on all numeric
  metrics + `tau_w`, and `config_hash` matches exactly. With a single machine's data there is
  nothing to compare yet (`shared fit-key(s): 0`) — that is expected, not a failure.
- **`AGREEMENT: FAIL`** — a flagged key exceeded tolerance (or a `config_hash` mismatch). Inspect
  the two machines + `max|d|`: a `config_hash` mismatch means different inputs/config (reconcile
  first); a tiny numeric `max|d|` just above tolerance across OS is usually BLAS/FP ordering —
  widen nothing silently, investigate.
- The bit-equal fingerprints (`config.hash16` / `feature_spec.hash16`) remain the handshake's
  job; this notebook is the downstream, tolerance-based agreement gate over actual results.